# CineData Analytics — Bronze to Silver

Este notebook é responsável pela transformação dos dados da camada Bronze para a camada Silver.

Nesta etapa são aplicados:
- padronização dos nomes das colunas;
- correção de tipos;
- tratamento de valores nulos e inválidos;
- deduplicação;
- regras de qualidade dos dados;
- enriquecimento e criação de colunas derivadas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

## 1. Informações dos filmes

Tratamento dos dados descritivos dos filmes, incluindo:
- deduplicação pela carga mais recente;
- padronização e tradução do status;
- conversão robusta das datas de lançamento;
- correção da tipagem;
- criação do ano de lançamento.

In [0]:
df_info_bronze = spark.table("bronze.tb_movies_info")

display(df_info_bronze.limit(10))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-18T17:51:08.262Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-18T17:51:08.262Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-18T17:51:08.262Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-18T17:51:08.262Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-18T17:51:08.262Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-09-18T17:51:08.262Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-09-18T17:51:08.262Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-09-18T17:51:08.262Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

In [0]:
df_info_bronze.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
janela_filme = (
    Window
    .partitionBy("id")
    .orderBy(F.col("ingestion_datetime").desc())
)

df_info_deduplicado = (
    df_info_bronze
    .withColumn(
        "row_number",
        F.row_number().over(janela_filme)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

In [0]:
df_info_tratado = (
    df_info_deduplicado

    .withColumn(
        "status_normalizado",
        F.lower(
            F.trim(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("status"),
                        r"[-_]+",
                        " "
                    ),
                    r"\s+",
                    " "
                )
            )
        )
    )

    .withColumn(
        "status_filme",
        F.when(F.col("status_normalizado") == "released", "Lançado")
         .when(F.col("status_normalizado") == "post production", "Pós-Produção")
         .when(F.col("status_normalizado") == "in production", "Em Produção")
         .when(F.col("status_normalizado") == "planned", "Planejado")
         .when(F.col("status_normalizado") == "rumored", "Rumores")
         .when(F.col("status_normalizado") == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )
)

In [0]:
df_info_tratado = (
    df_info_tratado
    .withColumn(
        "data_lancamento",
        F.coalesce(
            F.expr("try_to_date(trim(release_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(release_date), 'MM-dd-yyyy')"),
            F.expr("try_to_date(trim(release_date), 'dd/MM/yyyy')")
        )
    )
)

In [0]:
df_info_silver = (
    df_info_tratado
    .select(
        F.col("id").cast("string").alias("id_filme"),

        F.trim(F.col("title")).alias("titulo"),

        F.trim(F.col("original_title")).alias("titulo_original"),

        F.col("data_lancamento"),

        F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),

        F.trim(F.col("original_language")).alias("idioma_original"),

        F.col("status_filme"),

        F.trim(F.col("overview")).alias("sinopse"),

        F.trim(F.col("tagline")).alias("frase_divulgacao")
    )

    .withColumn(
        "ano_lancamento",
        F.year(F.col("data_lancamento"))
    )
)

In [0]:
display(df_info_silver.limit(20))

df_info_silver.printSchema()

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
14564,Rings,Rings,2017-02-01,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2017
32471,Mixtape,Mixtape,2021-12-03,94,en,Lançado,null,null,2021
38258,Grizzly II: Revenge,Grizzly II: Revenge,2020-02-17,74,en,Lançado,\All hell breaks loose when a giant grizzly,reacting to the slaughter of her cubs by poachers,2020
38492,Billy Joel - Live at Yankee Stadium,Billy Joel Live at Yankee Stadium,2022-06-22,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.,null,2022
38700,Bad Boys for Life,Bad Boys for Life,2020-01-15,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel.",Ride together. Die together.,2020
42018,The Horse Thief,盗马贼,2019-03-19,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter.",null,2019
42330,Monkey Magic,大闹西游,2018-09-22,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3",null,2018
43074,Ghostbusters,Ghostbusters,2016-07-14,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat.",Who You Gonna Call?,2016
45033,20 Seconds of Joy,20 Seconds of Joy,2018-01-01,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null,2018
46983,The Song of Styrene,Le Chant du styrène,2022-05-23,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.,null,2022


root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)



In [0]:
print("Quantidade após deduplicação:", df_info_silver.count())

print(
    "IDs duplicados:",
    df_info_silver
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Datas não convertidas:",
    df_info_silver
    .filter(F.col("data_lancamento").isNull())
    .count()
)

display(
    df_info_silver
    .groupBy("status_filme")
    .count()
    .orderBy(F.desc("count"))
)

Quantidade após deduplicação: 97879
IDs duplicados: 0
Datas não convertidas: 64


status_filme,count
Lançado,96463
Pós-Produção,701
Em Produção,604
Não Informado,64
Planejado,47


In [0]:
(
    df_info_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_info_filmes")
)

print("silver.tb_info_filmes criada com sucesso.")

silver.tb_info_filmes criada com sucesso.


In [0]:
print(
    "Bronze:",
    spark.table("bronze.tb_movies_info").count()
)

print(
    "Silver:",
    spark.table("silver.tb_info_filmes").count()
)

Bronze: 106930
Silver: 97879


## 2. Dados financeiros dos filmes

Tratamento das métricas financeiras dos filmes, incluindo:

- higienização de orçamento e receita;
- tratamento de valores ausentes ou inválidos;
- conversão para tipo decimal;
- conversão dos valores de USD para BRL;
- cálculo de lucro em USD e BRL;
- cálculo da margem de lucro percentual.

In [0]:
df_financeiro_bronze = spark.table("bronze.tb_movies_financials")

display(df_financeiro_bronze.limit(20))
df_financeiro_bronze.printSchema()

id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-09-18T17:51:13.354Z
299536,300000000,2052415039,2026-09-18T17:51:13.354Z
299534,356000000,2800000000,2026-09-18T17:51:13.354Z
475557,55000000,1074458282,2026-09-18T17:51:13.354Z
271110,250000000,Não Informado,2026-09-18T17:51:13.354Z
284054,200000000,1349926083,2026-09-18T17:51:13.354Z
284052,180000000,676343174,2026-09-18T17:51:13.354Z
315635,175000000,880166924,2026-09-18T17:51:13.354Z
283995,200000000,863756051,2026-09-18T17:51:13.354Z
297761,175000000,746846894,2026-09-18T17:51:13.354Z


root
 |-- id: integer (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
def limpar_valor_monetario(nome_coluna):

    valor = F.upper(
        F.trim(
            F.col(nome_coluna).cast("string")
        )
    )

    # Remove símbolos de moeda, texto USD e espaços.
    valor = F.regexp_replace(valor, r"USD|\$|\s", "")

    # Remove separadores de milhar com vírgula.
    valor = F.regexp_replace(valor, ",", "")

    # Trata números escritos com pontos de milhar:
    # exemplo: 1.000.000 -> 1000000
    valor = F.when(
        valor.rlike(r"^\d{1,3}(\.\d{3})+$"),
        F.regexp_replace(valor, r"\.", "")
    ).otherwise(valor)

    # Identifica abreviações como 10K ou 5M.
    multiplicador = (
        F.when(valor.rlike(r"K$"), F.lit(1_000))
         .when(valor.rlike(r"M$"), F.lit(1_000_000))
         .otherwise(F.lit(1))
    )

    # Remove K/M antes da conversão numérica.
    numero_texto = F.regexp_replace(valor, r"[KM]$", "")

    # Só tenta converter strings que realmente representam números.
    numero = (
        F.when(
            numero_texto.rlike(r"^-?\d+(\.\d+)?$"),
            numero_texto.cast("double")
        )
        .otherwise(F.lit(None).cast("double"))
    )

    valor_final = numero * multiplicador

    # Valores zero ou negativos devem ser tratados como ausentes.
    return (
        F.when(valor_final > 0, valor_final)
        .otherwise(F.lit(None))
        .cast("decimal(18,2)")
    )

In [0]:
df_financeiro_tratado = (
    df_financeiro_bronze
    .select(
        F.col("id").cast("string").alias("id_filme"),

        limpar_valor_monetario("budget").alias("orcamento_usd"),

        limpar_valor_monetario("revenue").alias("receita_usd")
    )
)

In [0]:
display(df_financeiro_tratado.limit(30))

id_filme,orcamento_usd,receita_usd
293660,58000000.00,null
299536,300000000.00,2052415039.00
299534,356000000.00,2800000000.00
475557,55000000.00,1074458282.00
271110,250000000.00,null
284054,200000000.00,1349926083.00
284052,180000000.00,676343174.00
315635,175000000.00,880166924.00
283995,200000000.00,863756051.00
297761,175000000.00,746846894.00


In [0]:
df_cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")

cotacao_atual = (
    df_cotacao_bronze
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()["cotacaoCompra"]
)

print(f"Cotação utilizada: R$ {cotacao_atual:.4f} por US$ 1")

Cotação utilizada: R$ 5.1569 por US$ 1


In [0]:
df_financeiro_silver = (
    df_financeiro_tratado

    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(cotacao_atual))
        .cast("decimal(18,2)")
    )

    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(cotacao_atual))
        .cast("decimal(18,2)")
    )
)

In [0]:
df_financeiro_silver = (
    df_financeiro_silver

    .withColumn(
        "lucro_usd",
        F.when(
            F.col("orcamento_usd").isNotNull()
            & F.col("receita_usd").isNotNull(),

            F.col("receita_usd") - F.col("orcamento_usd")
        )
        .otherwise(F.lit(None))
        .cast("decimal(18,2)")
    )

    .withColumn(
        "lucro_brl",
        F.when(
            F.col("orcamento_brl").isNotNull()
            & F.col("receita_brl").isNotNull(),

            F.col("receita_brl") - F.col("orcamento_brl")
        )
        .otherwise(F.lit(None))
        .cast("decimal(18,2)")
    )
)

In [0]:
df_financeiro_silver = (
    df_financeiro_silver
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull()
            & (F.col("receita_usd") > 0)
            & F.col("lucro_usd").isNotNull(),

            F.round(
                (F.col("lucro_usd") / F.col("receita_usd")) * 100,
                2
            )
        )
        .otherwise(F.lit(None))
        .cast("decimal(10,2)")
    )
)

In [0]:
display(df_financeiro_silver.limit(30))

df_financeiro_silver.printSchema()

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
293660,58000000.00,null,299100200.00,null,null,null,null
299536,300000000.00,2052415039.00,1547070000.00,10584099114.62,1752415039.00,9037029114.62,85.38
299534,356000000.00,2800000000.00,1835856400.00,14439320000.00,2444000000.00,12603463600.00,87.29
475557,55000000.00,1074458282.00,283629500.00,5540873914.45,1019458282.00,5257244414.45,94.88
271110,250000000.00,null,1289225000.00,null,null,null,null
284054,200000000.00,1349926083.00,1031380000.00,6961433817.42,1149926083.00,5930053817.42,85.18
284052,180000000.00,676343174.00,928242000.00,3487834114.00,496343174.00,2559592114.00,73.39
315635,175000000.00,880166924.00,902457500.00,4538932810.38,705166924.00,3636475310.38,80.12
283995,200000000.00,863756051.00,1031380000.00,4454303579.40,663756051.00,3422923579.40,76.85
297761,175000000.00,746846894.00,902457500.00,3851414747.67,571846894.00,2948957247.67,76.57


root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(10,2) (nullable = true)



In [0]:
print(
    "Orçamentos válidos:",
    df_financeiro_silver
    .filter(F.col("orcamento_usd").isNotNull())
    .count()
)

print(
    "Receitas válidas:",
    df_financeiro_silver
    .filter(F.col("receita_usd").isNotNull())
    .count()
)

print(
    "Orçamentos inválidos <= 0:",
    df_financeiro_silver
    .filter(F.col("orcamento_usd") <= 0)
    .count()
)

print(
    "Receitas inválidas <= 0:",
    df_financeiro_silver
    .filter(F.col("receita_usd") <= 0)
    .count()
)

Orçamentos válidos: 8884
Receitas válidas: 3582
Orçamentos inválidos <= 0: 0
Receitas inválidas <= 0: 0


In [0]:
(
    df_financeiro_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_financeiro_filmes")
)

print("silver.tb_financeiro_filmes criada com sucesso.")

silver.tb_financeiro_filmes criada com sucesso.


## 3. Métricas de engajamento

Tratamento das métricas de popularidade e avaliações dos filmes, incluindo:

- correção de separadores decimais;
- conversão segura de tipos;
- tratamento de dados deslocados ou corrompidos;
- validação das notas no intervalo de 0 a 10;
- invalidação de popularidade e contagens negativas.

In [0]:
df_metricas_bronze = spark.table("bronze.tb_movies_metrics")

display(df_metricas_bronze.limit(30))
df_metricas_bronze.printSchema()

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-09-18T17:51:17.195Z
299536,"154,34",8.255,27713,8.4,1406782,2026-09-18T17:51:17.195Z
299534,91.756,8.263,23857,8.4,1484150,2026-09-18T17:51:17.195Z
475557,"54,522",8.168,23425,8.3,1723035,2026-09-18T17:51:17.195Z
271110,70.741,7.4,21541,7.8,947222,2026-09-18T17:51:17.195Z
284054,43.665,7.39,null,7.3,924922,2026-09-18T17:51:17.195Z
284052,70.535,7.427,20935,7.5,895880,2026-09-18T17:51:17.195Z
315635,65.88,7.345,20507,7.4,835116,2026-09-18T17:51:17.195Z
283995,67.553,7.624,20353,7.6,844767,2026-09-18T17:51:17.195Z
297761,35.356,5.909,20097,5.9,null,2026-09-18T17:51:17.195Z


root
 |-- id: integer (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
def try_double(nome_coluna):
    return F.expr(
        f"""
        try_cast(
            regexp_replace(
                trim(cast(`{nome_coluna}` as string)),
                ',',
                '.'
            )
            AS DOUBLE
        )
        """
    )


def try_int(nome_coluna):
    return F.expr(
        f"""
        try_cast(
            trim(cast(`{nome_coluna}` as string))
            AS INT
        )
        """
    )

In [0]:
df_metricas_silver = (
    df_metricas_bronze
    .select(
        F.col("id").cast("string").alias("id_filme"),

        try_double("popularity").alias("popularidade"),

        try_double("vote_average").alias("nota_media_tmdb"),

        try_int("vote_count").alias("qtd_votos_tmdb"),

        try_double("averageRating").alias("nota_media_imdb"),

        try_int("numVotes").alias("qtd_votos_imdb")
    )
)

In [0]:
df_metricas_silver = (
    df_metricas_silver

    .withColumn(
        "popularidade",
        F.when(
            F.col("popularidade") >= 0,
            F.col("popularidade")
        )
    )

    .withColumn(
        "nota_media_tmdb",
        F.when(
            F.col("nota_media_tmdb").between(0, 10),
            F.col("nota_media_tmdb")
        )
    )

    .withColumn(
        "nota_media_imdb",
        F.when(
            F.col("nota_media_imdb").between(0, 10),
            F.col("nota_media_imdb")
        )
    )

    .withColumn(
        "qtd_votos_tmdb",
        F.when(
            F.col("qtd_votos_tmdb") >= 0,
            F.col("qtd_votos_tmdb")
        )
    )

    .withColumn(
        "qtd_votos_imdb",
        F.when(
            F.col("qtd_votos_imdb") >= 0,
            F.col("qtd_votos_imdb")
        )
    )
)

In [0]:
display(df_metricas_silver.limit(30))

df_metricas_silver.printSchema()

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735,7.606,28894,8.0,1270339
299536,154.34,8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,54.522,8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


root
 |-- id_filme: string (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



In [0]:
print(
    "Notas TMDB fora do intervalo:",
    df_metricas_silver
    .filter(
        (F.col("nota_media_tmdb") < 0)
        | (F.col("nota_media_tmdb") > 10)
    )
    .count()
)

print(
    "Notas IMDb fora do intervalo:",
    df_metricas_silver
    .filter(
        (F.col("nota_media_imdb") < 0)
        | (F.col("nota_media_imdb") > 10)
    )
    .count()
)

print(
    "Popularidades negativas:",
    df_metricas_silver
    .filter(F.col("popularidade") < 0)
    .count()
)

print(
    "Contagens TMDB negativas:",
    df_metricas_silver
    .filter(F.col("qtd_votos_tmdb") < 0)
    .count()
)

print(
    "Contagens IMDb negativas:",
    df_metricas_silver
    .filter(F.col("qtd_votos_imdb") < 0)
    .count()
)

Notas TMDB fora do intervalo: 0
Notas IMDb fora do intervalo: 0
Popularidades negativas: 0
Contagens TMDB negativas: 0
Contagens IMDb negativas: 0


In [0]:
(
    df_metricas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_metricas_engajamento")
)

print("silver.tb_metricas_engajamento criada com sucesso.")

silver.tb_metricas_engajamento criada com sucesso.


## 4. Avaliações dos usuários

Tratamento das avaliações realizadas pelos usuários, incluindo:

- remoção de avaliações integralmente duplicadas;
- conversão segura da nota;
- validação da escala de 0 a 10;
- preenchimento de comentários ausentes com o texto "Sem comentário".

In [0]:
df_avaliacoes_bronze = spark.table("bronze.tb_movies_reviews")

display(df_avaliacoes_bronze.limit(30))
df_avaliacoes_bronze.printSchema()

id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-18T17:51:25.341Z
637007,Lucas Reis 602,3.9,null,2026-09-18T17:51:25.341Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-18T17:51:25.341Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-18T17:51:25.341Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-18T17:51:25.341Z
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo.",2026-09-18T17:51:25.341Z
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo.",2026-09-18T17:51:25.341Z
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.,2026-09-18T17:51:25.341Z
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.,2026-09-18T17:51:25.341Z
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular.",2026-09-18T17:51:25.341Z


root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: double (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
df_avaliacoes_silver = (
    df_avaliacoes_bronze

    .select(
        F.col("id").cast("string").alias("id_filme"),

        F.trim(F.col("nome")).alias("nome_usuario"),

        F.expr(
            """
            try_cast(
                trim(cast(nota as string))
                AS DOUBLE
            )
            """
        ).alias("nota_usuario"),

        F.col("comentario").alias("comentario_usuario")
    )

    .withColumn(
        "nota_usuario",
        F.when(
            F.col("nota_usuario").between(0, 10),
            F.col("nota_usuario")
        )
    )

    .withColumn(
        "comentario_usuario",
        F.when(
            F.col("comentario_usuario").isNull()
            | (F.trim(F.col("comentario_usuario")) == ""),
            F.lit("Sem comentário")
        )
        .otherwise(F.trim(F.col("comentario_usuario")))
    )

    .dropDuplicates(
        [
            "id_filme",
            "nome_usuario",
            "nota_usuario",
            "comentario_usuario"
        ]
    )
)

In [0]:
display(df_avaliacoes_silver.limit(30))

df_avaliacoes_silver.printSchema()

id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


root
 |-- id_filme: string (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)



In [0]:
print(
    "Notas fora do intervalo:",
    df_avaliacoes_silver
    .filter(
        (F.col("nota_usuario") < 0)
        | (F.col("nota_usuario") > 10)
    )
    .count()
)

print(
    "Comentários nulos ou vazios:",
    df_avaliacoes_silver
    .filter(
        F.col("comentario_usuario").isNull()
        | (F.trim(F.col("comentario_usuario")) == "")
    )
    .count()
)

print(
    "Avaliações duplicadas:",
    df_avaliacoes_silver
    .groupBy(
        "id_filme",
        "nome_usuario",
        "nota_usuario",
        "comentario_usuario"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Notas fora do intervalo: 0
Comentários nulos ou vazios: 0
Avaliações duplicadas: 0


In [0]:
(
    df_avaliacoes_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_avaliacoes_usuarios")
)

print("silver.tb_avaliacoes_usuarios criada com sucesso.")

silver.tb_avaliacoes_usuarios criada com sucesso.


## 5. Gêneros dos filmes

Tratamento dos gêneros associados aos filmes, incluindo:

- normalização dos diferentes separadores;
- desmembramento dos múltiplos gêneros com `split` e `explode`;
- remoção de valores vazios e resíduos;
- validação dos gêneros pertencentes ao domínio esperado;
- eliminação de registros duplicados.

In [0]:
df_credits_bronze = spark.table("bronze.tb_credits_and_tags")

display(
    df_credits_bronze
    .select("id", "genres")
    .limit(30)
)

id,genres
293660,"Action, Adventure, Comedy"
299536,"Adventure, Action, Science Fiction"
299534,"Adventure, Science Fiction, Action"
475557,"Crime, Thriller, Drama"
271110,"Adventure, Action, Science Fiction"
284054,"Action, Adventure, Science Fiction"
284052,"Action, Adventure, Fantasy"
315635,"Action, Adventure, Science Fiction, Drama"
283995,"Science Fiction, Adventure, Action"
297761,Action|Adventure|Fantasy


In [0]:
df_generos = (
    df_credits_bronze

    .select(
        F.col("id").cast("string").alias("id_filme"),

        F.regexp_replace(
            F.col("genres"),
            r"[;|]",
            ","
        ).alias("genres_normalizados")
    )

    .withColumn(
        "genero",
        F.explode(
            F.split(
                F.col("genres_normalizados"),
                ","
            )
        )
    )

    .withColumn(
        "genero",
        F.trim(F.col("genero"))
    )

    .drop("genres_normalizados")
)

In [0]:
generos_validos = [
    "Action",
    "Adventure",
    "Animation",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Family",
    "Fantasy",
    "History",
    "Horror",
    "Music",
    "Mystery",
    "Romance",
    "Science Fiction",
    "TV Movie",
    "Thriller",
    "War",
    "Western"
]

In [0]:
df_generos_silver = (
    df_generos

    .filter(
        F.col("genero").isNotNull()
        & (F.trim(F.col("genero")) != "")
    )

    .filter(
        F.col("genero").isin(generos_validos)
    )

    .dropDuplicates(
        ["id_filme", "genero"]
    )
)

In [0]:
display(df_generos_silver.limit(30))

df_generos_silver.printSchema()

id_filme,genero
329865,Mystery
269149,Animation
429617,Science Fiction
330459,Action
127380,Family
332562,Romance
291805,Action
293167,Action
381284,History
454626,Action


root
 |-- id_filme: string (nullable = true)
 |-- genero: string (nullable = false)



In [0]:
print(
    "Registros com gênero vazio:",
    df_generos_silver
    .filter(
        F.col("genero").isNull()
        | (F.trim(F.col("genero")) == "")
    )
    .count()
)

print(
    "Relações filme-gênero duplicadas:",
    df_generos_silver
    .groupBy("id_filme", "genero")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(
    df_generos_silver
    .groupBy("genero")
    .count()
    .orderBy(F.desc("count"))
)

Registros com gênero vazio: 0
Relações filme-gênero duplicadas: 0


genero,count
Drama,32647
Documentary,19235
Comedy,18824
Thriller,10402
Horror,9854
Romance,7712
Action,6100
Crime,4781
Animation,4518
TV Movie,4113


In [0]:
(
    df_generos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_generos")
)

print("silver.tb_generos criada com sucesso.")

silver.tb_generos criada com sucesso.


## 6. Pessoas e empresas

Consolidação das pessoas e empresas relacionadas aos filmes em uma dimensão unificada.

Nesta etapa:
- os valores múltiplos são separados e explodidos;
- atores, diretores, roteiristas e produtoras são consolidados;
- cada entidade recebe seu respectivo tipo de atuação;
- nomes são padronizados;
- resíduos e valores inválidos são removidos;
- registros duplicados são eliminados.

In [0]:
def tratar_entidades(df, coluna_origem, tipo_entidade):

    return (
        df
        .select(
            F.col("id").cast("string").alias("id_filme"),

            F.explode(
                F.split(
                    F.regexp_replace(
                        F.col(coluna_origem),
                        r"[;|]",
                        ","
                    ),
                    ","
                )
            ).alias("nome_entidade")
        )

        .withColumn(
            "nome_entidade",
            F.trim(F.col("nome_entidade"))
        )

        # Mantém apenas valores não vazios e que possuam texto.
        .filter(
            F.col("nome_entidade").isNotNull()
            & (F.col("nome_entidade") != "")
            & F.col("nome_entidade").rlike(r".*[A-Za-zÀ-ÿ].*")
        )

        # Evita resíduos muito extensos provenientes de column shift.
        .filter(
            F.length(F.col("nome_entidade")) <= 120
        )

        .withColumn(
            "nome_entidade",
            F.initcap(F.col("nome_entidade"))
        )

        .withColumn(
            "tipo_entidade",
            F.lit(tipo_entidade)
        )
    )

In [0]:
df_atores = tratar_entidades(
    df_credits_bronze,
    "cast",
    "Ator"
)

df_diretores = tratar_entidades(
    df_credits_bronze,
    "directors",
    "Diretor"
)

df_roteiristas = tratar_entidades(
    df_credits_bronze,
    "writers",
    "Roteirista"
)

df_produtoras = tratar_entidades(
    df_credits_bronze,
    "production_companies",
    "Produtora"
)

In [0]:
df_pessoas_empresas_silver = (
    df_atores

    .unionByName(df_diretores)

    .unionByName(df_roteiristas)

    .unionByName(df_produtoras)

    .dropDuplicates(
        [
            "id_filme",
            "nome_entidade",
            "tipo_entidade"
        ]
    )
)

In [0]:
display(df_pessoas_empresas_silver.limit(40))

df_pessoas_empresas_silver.printSchema()

id_filme,nome_entidade,tipo_entidade
299536,Don Cheadle,Ator
263115,Hugh Jackman,Ator
354912,Anthony Gonzalez,Ator
419430,Daniel Kaluuya,Ator
424694,Mike Myers,Ator
374720,Mark Rylance,Ator
321612,Ewan Mcgregor,Ator
335983,Tom Hardy,Ator
363088,Evangeline Lilly,Ator
363088,Randall Park,Ator


root
 |-- id_filme: string (nullable = true)
 |-- nome_entidade: string (nullable = false)
 |-- tipo_entidade: string (nullable = false)



In [0]:
print(
    "Entidades vazias:",
    df_pessoas_empresas_silver
    .filter(
        F.col("nome_entidade").isNull()
        | (F.trim(F.col("nome_entidade")) == "")
    )
    .count()
)

print(
    "Registros duplicados:",
    df_pessoas_empresas_silver
    .groupBy(
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(
    df_pessoas_empresas_silver
    .groupBy("tipo_entidade")
    .count()
    .orderBy(F.desc("count"))
)

Entidades vazias: 0
Registros duplicados: 0


tipo_entidade,count
Ator,546869
Roteirista,132273
Produtora,120025
Diretor,102790


In [0]:
(
    df_pessoas_empresas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_pessoas_empresas")
)

print("silver.tb_pessoas_empresas criada com sucesso.")

silver.tb_pessoas_empresas criada com sucesso.


## 7. Cotação do dólar

Tratamento do histórico de cotações do dólar, incluindo:

- conversão da data da cotação;
- criação de uma série temporal diária contínua;
- preenchimento dos dias sem cotação;
- aplicação de Forward Fill utilizando a última cotação disponível.

In [0]:
df_cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")

display(df_cotacao_bronze.orderBy("dataHoraCotacao"))
df_cotacao_bronze.printSchema()

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.0912,2026-09-11 13:07:22.532196,2026-09-18T17:30:22.393Z
5.169,2026-09-14 13:10:08.144425,2026-09-18T17:30:22.393Z
5.1484,2026-09-15 13:09:19.199664,2026-09-18T17:30:22.393Z
5.152,2026-09-16 13:05:30.35873,2026-09-18T17:30:22.393Z
5.1515,2026-09-17 13:03:21.858212,2026-09-18T17:30:22.393Z
5.1569,2026-09-18 13:03:34.742036,2026-09-18T17:30:22.393Z


root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
df_cotacao_tratado = (
    df_cotacao_bronze

    .select(
        F.to_date(
            F.substring(F.col("dataHoraCotacao"), 1, 10),
            "yyyy-MM-dd"
        ).alias("data_cotacao"),

        F.col("cotacaoCompra")
        .cast("decimal(10,4)")
        .alias("cotacao_compra"),

        F.col("dataHoraCotacao"),

        F.col("ingestion_datetime")
    )
)

In [0]:
janela_cotacao = (
    Window
    .partitionBy("data_cotacao")
    .orderBy(
        F.col("dataHoraCotacao").desc(),
        F.col("ingestion_datetime").desc()
    )
)

df_cotacao_diaria = (
    df_cotacao_tratado

    .withColumn(
        "row_number",
        F.row_number().over(janela_cotacao)
    )

    .filter(F.col("row_number") == 1)

    .select(
        "data_cotacao",
        "cotacao_compra"
    )
)

In [0]:
limites = (
    df_cotacao_diaria
    .agg(
        F.min("data_cotacao").alias("data_inicio"),
        F.max("data_cotacao").alias("data_fim")
    )
    .first()
)

data_inicio = limites["data_inicio"]
data_fim = limites["data_fim"]

print(f"Período: {data_inicio} até {data_fim}")

Período: 2026-09-11 até 2026-09-18


In [0]:
df_calendario = (
    spark
    .range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_inicio),
                F.lit(data_fim),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

In [0]:
df_cotacao_completa = (
    df_calendario

    .join(
        df_cotacao_diaria,
        on="data_cotacao",
        how="left"
    )

    .orderBy("data_cotacao")
)

In [0]:
janela_forward_fill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

In [0]:
df_cotacao_silver = (
    df_cotacao_completa

    .withColumn(
        "cotacao_compra",
        F.last(
            F.col("cotacao_compra"),
            ignorenulls=True
        ).over(janela_forward_fill)
    )

    .select(
        "data_cotacao",
        "cotacao_compra"
    )

    .orderBy("data_cotacao")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(df_cotacao_silver)

df_cotacao_silver.printSchema()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-11,5.0912
2026-09-12,5.0912
2026-09-13,5.0912
2026-09-14,5.1690
2026-09-15,5.1484
2026-09-16,5.1520
2026-09-17,5.1515
2026-09-18,5.1569


root
 |-- data_cotacao: date (nullable = false)
 |-- cotacao_compra: decimal(10,4) (nullable = true)



In [0]:
print(
    "Dias na Bronze:",
    df_cotacao_diaria.count()
)

print(
    "Dias na Silver:",
    df_cotacao_silver.count()
)

print(
    "Cotações nulas após Forward Fill:",
    df_cotacao_silver
    .filter(F.col("cotacao_compra").isNull())
    .count()
)

In [0]:
(
    df_cotacao_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_cotacao_dolar")
)

print("silver.tb_cotacao_dolar criada com sucesso.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


silver.tb_cotacao_dolar criada com sucesso.


## Validação final da camada Silver

Verificação das tabelas geradas após os processos de limpeza, padronização e transformação.

In [0]:
tabelas_silver = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in tabelas_silver:
    quantidade = spark.table(f"silver.{tabela}").count()
    print(f"silver.{tabela}: {quantidade} registros")

silver.tb_info_filmes: 97879 registros
silver.tb_financeiro_filmes: 106165 registros
silver.tb_metricas_engajamento: 107364 registros
silver.tb_avaliacoes_usuarios: 32412 registros
silver.tb_generos: 141965 registros
silver.tb_pessoas_empresas: 901957 registros
silver.tb_cotacao_dolar: 8 registros
